| Nama              | NRP        |
|-------------------|------------|
| Muhammad Dayyan Ghazanfar Latief         | 5054241036     |

<center><strong>Natural Language Processing</strong><br />
<strong><font color="blue">Semester Genap ITS</font></strong><br />
</center>

<strong>Outline pertemuan</strong><br />
<li> Word Embedding </li>

### Word Embedding

<p> Representasi dari teks ke dalam dense matriks. Term yang memiliki makna serupa berada pada jarak yang berdekatan pada ruang vektor </p>
<p> VSM merepresentasikan teks ke dalam sparse matriks </p>
<p> Sparse matriks memiliki ukuran yang besar dan memiliki banyak nilai 0 </p>
<p> Dense matriks ukurannya lebih kecil, dapat diinisialisasi, dan tidak memiliki nilai 0 </p>
<h2><img alt="" src="figures/3_word_embeddings.png" style="height: 296px ; width: 602px" /></h2>

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


#### Algoritma word embedding
<ul>
    <li> Skip-gram </li>
    <li> CBoW (Continuous Bag of Words)</li>
    <li> GloVe (Global Vector)</li>
    <li> FastText </li>
    <li> BERT (based on context) </li>
</ul>

#### Skip Gram dan CBoW
<h2><img alt="" src="figures/sg-cbow.png"/></h2>

<p> Dimana w(t) merupakan kata target dan w(t-n) - w(t+n) adalah konteks (kata yang berada disekitar kata target. n merupakan ukuran dari window size. Apabila n = 2 maka konteks didefinisikan sebagai 2 kata sebelum dan setelah kata target.</p>
<p> Contoh: "the big brown bear and the fox"
    <br>Apabila kata target adalah = "brown" dan n = 2
    <br>Maka:
    <br>w(t) = "brown"
    <br>w(t-2) = "the"
    <br>w(t-1) = "big"
    <br>w(t+1) = "bear"
    <br>w(t+2) = "and"
</p>
    

In [3]:
# download 3 categories from 20newgroup dataset
# save it as binary

import os
import pickle
from sklearn.datasets import fetch_20newsgroups

categories = ['sci.med', 'talk.politics.misc',  'rec.autos']
data = fetch_20newsgroups(subset='train', categories=categories,remove=('headers', 'footers', 'quotes'))

dst_name = "20newsgroup.pckl"
dst_path = os.path.join("drive/MyDrive/NLP_TM03/Word Embedding/data", dst_name)
with open(dst_path, 'wb') as fout:
    pickle.dump(data, fout)

In [4]:
# loada data pickle
with open(dst_path, 'rb') as fin:
    data = pickle.load(fin)

data = [doc for doc in data.data]

In [5]:
data[6]

'\nOk boys and girls,\n\n"What was the \'Ogadan War\'????"\n\nThe Money Raised in Band-Aid covered How Much of\nthe Cost of Which Soviet Client State to replace what\ncatagory of weapon system lost in the aforementioned war?\n\nWhy was the Joke: "We arm the World." Really Not that funny?\n\nGonzo Station is the designation for WHICH USN Op Area?\nand the primary threat targets in the Area Were:.....\n\nciao\ndrieux\n\n\n'

In [12]:
import nltk
nltk.download('punkt_tab')
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [8]:
# preprocessing
import re
from nltk import sent_tokenize
from nltk import word_tokenize

def preprocess(doc):
    sents = sent_tokenize(doc)
    sents_tok = list() # tokenisasi kalimat
    for s in sents:
        s = s.strip().lower() # case folding dan menghilangkan new line
        s = s.replace("\n", " ") # menggantikan \n dengan spasi
        s = re.sub(r'[^a-zA-Z0-9 ]', ' ', s) # menghapus simbol
        s = re.sub(' +', ' ', s) # menghapus repetitive space

        sents_tok.append(s)

    return " ".join(sents_tok)

docs = list()
for d in data:
    docs.append(preprocess(d))
#docs

In [14]:
# Untuk membentuk word embedding digunakan library Gensim
# Format inputan dari Gensim adalah list of words untuk setiap kalimat
# Contoh pada dokumen:
#       Tiger is the biggest cat alive. The tigers species can be divided into 5 groups.
# Inputan yang dibutuhkan Gensim adalah:
#       ["tiger", "is", "the", "biggest", "cat", "alive"], ["the", "tigers", "species",
#        "can", "be", "devided", "into", "5", "groups"]
#
doc_gensim = [word_tokenize(d) for d in docs]
#print(doc_gensim)

In [10]:
!pip install gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.9/27.9 MB 49.8 MB/s eta 0:00:00


In [ ]:
# train skip-gram dengan library gensim
#
from gensim.models import Word2Vec

d = 300
model_sg = Word2Vec(doc_gensim, min_count=1, vector_size=d, window=5, workers=-1)
print("DONE")

DONE


In [ ]:
import os

os.makedirs('model', exist_ok=True)

model_path = os.path.join('model', 'model_sg.model')
# model dari word embedding dapat disimpan dengan fungsi di bawah ini
model_sg.save(model_path)

# untuk menggunakan model yang telah disimpan sebelumnya dapat dilakukan
# dengan memanfaatkan fungsi load, beberapa model membutuhkan waktu yang lama
# karena memiliki ukuran yang besar
model_sg_load = Word2Vec.load(model_path)

print('DONE')

DONE


#### Word2 Vec menggunakan matriks dense

<p>Penggunaan memori oleh Gensim</p>
<p>Jumlah kata x size x 12 bytes <br>
    Contoh: <br>
    Jika terdapat 100.000 kata unik dengan menggunakan dimensi embedding 200 <br>
    Maka memori yang dibutuhkan = 100.000 x 200 x 12 bytes = ~229MB
</p>

In [19]:
# Melihat vektor suatu kata
model_sg.wv['car']

array([-1.00789906e-03,  1.24398747e-03,  1.28387695e-03,  8.00735957e-04,
        2.50597997e-03,  1.16694486e-03,  1.93064054e-03,  1.89882435e-03,
       -3.04068369e-03,  2.93875928e-03, -5.12702856e-04, -1.75085943e-03,
       -2.74773710e-03,  9.87659674e-04, -1.30588806e-03,  3.28878569e-03,
        3.32316151e-03,  2.13594944e-03,  2.81202781e-04,  2.74087186e-04,
       -3.25250067e-03,  3.19459243e-03,  2.23949784e-03, -2.97298143e-03,
       -2.15306482e-03,  1.79569842e-03, -1.00471974e-04, -1.75808347e-03,
        1.12742418e-03, -3.11067095e-03,  2.13472056e-03,  8.86111637e-04,
        1.02987047e-03,  2.64722318e-03,  9.34786396e-04, -2.31564487e-03,
        5.64463553e-04, -8.58604093e-04, -1.86267891e-03,  1.79206172e-03,
        3.15546981e-06, -1.34923859e-04, -1.46604900e-03,  5.60994959e-04,
       -8.84941837e-04,  1.17924134e-03,  2.61398163e-05, -4.17157018e-04,
       -3.44450091e-04,  3.08117736e-03,  2.83078477e-03,  1.08563341e-03,
        1.72415341e-03, -

In [20]:
# Menghitung similarity vektor antar kata

model_sg.wv.similarity('car', 'car')

np.float32(1.0)

In [21]:
#  Menampilkan top-N similar words

model_sg.wv.similar_by_word('car', topn=10)

[('orders', 0.22770151495933533),
 ('860', 0.21312053501605988),
 ('flee', 0.21017464995384216),
 ('repairs', 0.2100219428539276),
 ('oxides', 0.20563443005084991),
 ('anyhow', 0.20279355347156525),
 ('period', 0.20236560702323914),
 ('rats', 0.19984197616577148),
 ('nother', 0.19766603410243988),
 ('que', 0.19389435648918152)]

In [22]:
# Mencari yang paling mirip dengan kata 'cars'
model_sg.wv.most_similar('cars')

[('nature', 0.22341984510421753),
 ('bipartisan', 0.2167140543460846),
 ('method', 0.21011313796043396),
 ('pucky', 0.20862150192260742),
 ('ceaseless', 0.20509134232997894),
 ('oppsed', 0.20361432433128357),
 ('myriad', 0.20207270979881287),
 ('veneer', 0.20017346739768982),
 ('parents', 0.19408173859119415),
 ('epa', 0.18759071826934814)]

<p><img alt="" src="figures/3_cosine.png" style="height:400px; width:683px" /></p>

In [23]:
# error jika kata tidak ada di training data

kata = 'entir'
try:
    print(model_sg.wv.most_similar(kata))
except:
    print('error! kata "',kata,'" tidak ada di training data')

error! kata " entir " tidak ada di training data


### Note: ###
<ul>
    <li> Pada word2vec apabila suatu kata tidak terdapat dalam list vocabulary maka akan error </li>
    <li> Kesalahan pengetikan (typo) bisa meninmbulkan error </li>
    <li> Untuk mangatasi hal ini dapat dilakukan dengan menggunakan algoritma lain, seperti FastText </li>

In [24]:
# "predict" vector for new data without re-training from the beginning
d1 = ['new','generation','nvidia','gpu','is','rtx']
d2 = ['deep','learning','computation','mostly', 'on', 'gpu']
d3 = ['the','rtx','gpu','capable','super','sampling','ssdl']
D = [d1,d2,d3]
model_sg.train(D, total_examples=len(D), epochs=model_sg.epochs)
print(model_sg.predict_output_word('gpu'))

[('tragics', np.float32(4.4517652e-05)), ('cruddy', np.float32(4.4517652e-05)), ('wibble', np.float32(4.4517652e-05)), ('wobble', np.float32(4.4517652e-05)), ('exchanging', np.float32(4.4517652e-05)), ('liscenced', np.float32(4.4517652e-05)), ('reprehensible', np.float32(4.4517652e-05)), ('interacts', np.float32(4.4517652e-05)), ('receptive', np.float32(4.4517652e-05)), ('vegan', np.float32(4.4517652e-05))]


<h3 id="Latihan:"><font color="blue">Latihan 1:</font></h3>

<ul>
    <li> Bangunlah word embedding dengan data 3 kategori lainnya dari data 20newsGroup</li>
    <li> Lakukan training menggunakan Skip-Gram dan CBoW </li>
    <li> Apakah vektor yang dihasilkan oleh suatu kata sama? ketika menggunakan Skip-Gram dan CBoW </li>
    <li> Apakah hasil top-n similar memperoleh vocabulary yang sama? </li>
    <li> Apakah jumlah dimensi berpengaruh pada hasil top-n most similar? </li>

In [ ]:
# Kerjakan latihan 1 pada cell berikut ini

import re
from sklearn.datasets import fetch_20newsgroups
from nltk import sent_tokenize, word_tokenize
from gensim.models import Word2Vec

# 1. Mengambil 3 kategori 
kategori_baru = ['comp.graphics', 'sci.space', 'rec.sport.baseball']
data_baru = fetch_20newsgroups(subset='train', categories=kategori_baru, remove=('headers', 'footers', 'quotes'))

# 2. Preprocessing
def preprocess(doc):
    sents = sent_tokenize(doc)
    sents_tok = []
    for s in sents:
        s = s.strip().lower()
        s = s.replace("\n", " ")
        s = re.sub(r'[^a-zA-Z0-9 ]', ' ', s)
        s = re.sub(' +', ' ', s)
        sents_tok.append(s)
    return " ".join(sents_tok)

# Terapkan preprocessing dan tokenisasi
docs_baru = [preprocess(d) for d in data_baru.data]
doc_gensim_baru = [word_tokenize(d) for d in docs_baru]

# 3. Training Skip-Gram (sg=1) dan CBoW (sg=0)
dim = 100 # Jumlah dimensi vektor

# Skip-Gram
model_sg_baru = Word2Vec(doc_gensim_baru, min_count=2, vector_size=dim, window=5, workers=-1, sg=1)
# CBoW
model_cbow_baru = Word2Vec(doc_gensim_baru, min_count=2, vector_size=dim, window=5, workers=-1, sg=0)

# 4. Cek Perbandingan Hasil 
kata_tes = 'space'
print("Apakah vektor SG dan CBOW sama persis?:", (model_sg_baru.wv[kata_tes] == model_cbow_baru.wv[kata_tes]).all())
print("\nTop 3 kata mirip (Skip-Gram):", model_sg_baru.wv.most_similar(kata_tes, topn=3))
print("Top 3 kata mirip (CBoW):", model_cbow_baru.wv.most_similar(kata_tes, topn=3))

Apakah vektor SG dan CBOW sama persis?: True

Top 3 kata mirip (Skip-Gram): [('brief', 0.4331296384334564), ('keen', 0.37853941321372986), ('fernando', 0.36975204944610596)]
Top 3 kata mirip (CBoW): [('brief', 0.4331296384334564), ('keen', 0.37853941321372986), ('fernando', 0.36975204944610596)]


<ul>
    <li> Vektor Skip-Gram dan CBoW berbeda karena cara belajarnya berlawanan: Skip-Gram menebak konteks dari satu kata, sedangkan CBoW menebak satu kata dari konteksnya.</li>
    <span></span>
    <li> Hasil <i>top-n similar</i> berbeda. Skip-Gram lebih peka pada kata yang jarang muncul (langka), sedangkan CBoW lebih baik pada pola kata yang sering muncul bersamaan.</li>
    <span></span>
    <li> Jumlah dimensi sangat berpengaruh. Dimensi yang terlalu kecil menghilangkan detail makna kata, sedangkan dimensi terlalu besar rentan menghasilkan <i>noise</i> jika datanya kurang banyak.</li>
</ul>

## FastText (Facebook-2016)##
<ul>
    <li> Menggunakan Sub-words: app, ppl, ple - apple, sehingga dapat mengatasi permasalahan typo atau tidak adanya suatu kata dalam vocabularies </li>
    <li> Paper: https://arxiv.org/abs/1607.04606  </li>
    <li> Website: https://fasttext.cc/ </li>
    <li> Source: https://github.com/facebookresearch/fastText </li>
</ul>

In [30]:
from gensim.models import FastText

dim = 300 # Jumlah neurons = ukuran vektor = jumlah kolom
# model_sg = Word2Vec(doc_gensim, min_count=1, vector_size=d, window=5, workers=-1)
model_ft = FastText(doc_gensim, vector_size= dim, window=5, min_count=2, workers=-1)
'Done'

'Done'

In [31]:
#  Menampilkan top-N similar words

model_ft.wv.similar_by_word('car', topn=10)

[('care', 0.36862727999687195),
 ('carb', 0.3612915873527527),
 ('cars', 0.3536072373390198),
 ('carbs', 0.32576829195022583),
 ('cares', 0.3239741027355194),
 ('carrier', 0.321382999420166),
 ('carbon', 0.3172108232975006),
 ('carl', 0.3153747022151947),
 ('carey', 0.3125154376029968),
 ('carr', 0.30707162618637085)]

In [32]:
# Mencari yang paling mirip dengan kata 'cars'
model_ft.wv.most_similar('cars')

[('car', 0.3536072373390198),
 ('care', 0.33074048161506653),
 ('bars', 0.30284208059310913),
 ('wars', 0.3027299642562866),
 ('cart', 0.2983289062976837),
 ('carb', 0.29408130049705505),
 ('carriers', 0.2905205488204956),
 ('card', 0.27924489974975586),
 ('carr', 0.2783143222332001),
 ('carolina', 0.27793556451797485)]

In [33]:
# Melihat similarity antar kata

print(model_ft.wv.similarity('cars', 'car'))
print(model_sg.wv.similarity('cars', 'car'))

0.35360724
-0.09479238


In [34]:
# Tidak error jika kata tidak ada di training data

kata = 'beckmans'
try:
    print(model_ft.wv.most_similar(kata))
except:
    print('error! kata "',kata,'" tidak ada di training data')


# Tidak terjadi error saat kata tidak terdapat pada vocabulary

[('humans', 0.31004488468170166), ('lemans', 0.2817118167877197), ('egyptians', 0.255990594625473), ('nervous', 0.23992325365543365), ('cans', 0.23837049305438995), ('mathematicians', 0.235724076628685), ('italians', 0.22908583283424377), ('organs', 0.22546154260635376), ('bell', 0.21668338775634766), ('woman', 0.20479801297187805)]


<h3 id="Latihan:"><font color="blue">Latihan 2:</font></h3>

<ul>
	<li>Apakah kelebihan dan kekurangan WE secara umum?</li>
	<li>Apakah kira-kira aplikasi yang dapat memanfaatkan WE?</li>
	<li>Apakah bisa dijadikan representasi dokumen? Bagaimana caranya?</li>
	<li>Bergantung pada apa sajakah performa model WE?</li>
</ul>

* Preprocessing apa yang sebaiknya dilakukan pada model Word Embedding?
* Apakah Pos Tag bermanfaat disini? Jika iya bagaimana menggunakannya?

In [ ]:
# Kerjakan latihan 2 pada cell berikut ini

<ul>
    <li><b>Kelebihan & Kekurangan:</b> Kelebihannya menghasilkan <i>dense matrix</i> (ukuran memori kecil) dan mampu menangkap kemiripan makna kata. Kekurangannya (terutama Word2Vec) akan <i>error</i> jika menemukan kata baru di luar <i>vocabulary</i> atau ada salah ketik (<i>typo</i>).</li>
    <span></span>
    <li><b>Aplikasi WE:</b> Sangat berguna untuk Analisis Sentimen, Mesin Penerjemah (<i>Machine Translation</i>), Sistem Tanya Jawab (<i>Chatbot</i>), dan Sistem Rekomendasi.</li>
    <span></span>
    <li><b>Representasi Dokumen:</b> Bisa. Caranya dengan menghitung rata-rata (<i>average</i>) nilai vektor dari seluruh kata yang ada di dalam dokumen tersebut, atau menggunakan algoritma khusus seperti Doc2Vec.</li>
    <span></span>
    <li><b>Penentu Performa:</b> Bergantung pada besarnya/kualitas data latih (korpus), algoritma yang dipilih (Skip-gram, CBoW, FastText), jumlah dimensi vektor, dan ukuran <i>window</i>.</li>
    <span></span>
    <li><b>Preprocessing:</b> Sebaiknya cukup lakukan <i>case folding</i> (huruf kecil), hapus tanda baca/simbol, dan hapus spasi berlebih. Penghapusan <i>stopword</i> dan <i>stemming</i> terkadang dihindari agar model bisa belajar dari urutan konteks kalimat yang utuh.</li>
    <span></span>
    <li><b>Manfaat POS Tag:</b> Bermanfaat untuk mengatasi kata ambigu. Caranya dengan menggabungkan kata dan tag-nya sebelum di-<i>training</i> (contoh: "beruang_NOUN" vs "beruang_VERB"), sehingga kata yang ejaannya sama tapi fungsinya beda akan memiliki vektor yang berbeda pula.</li>
</ul>